# Алгоритм серых волков

**GWO** - Grey Wolves Algorithm

In [ ]:
import numpy as np
import osmnx as ox
import pandas as pd
import geopandas as gpd
import networkx as nx

import matplotlib.pyplot as plt

from graphs.algorithms import generate_grid_graph
from shapely.geometry import LineString, Polygon

import genesis as genesis

# Версии библиотек
print('OSMnx:', ox.__version__,
      '\nGeoPandas:', gpd.__version__,
      '\nPandas:', pd.__version__,
      '\ngenesis:', genesis.__version__,
      ) 

## Тестовый граф

In [ ]:
G = generate_grid_graph(n=100, m=100, step=100.0, speed=20, diagonals=True)
nx.set_edge_attributes(G, 'undefined', 'highway')

## Реализация GWO

### Реализация алгоритма GWO (Grey Wolf Optimizer)

Я успешно реализовал алгоритм серых волков (Grey Wolf Optimizer) для поиска оптимального размещения одного пожарного подразделения. Вот ключевые аспекты реализации:

#### Особенности реализации:

1. **Наследование**: Класс `BestNodeGWO` наследуется от `BestPointsBase`, как и другие алгоритмы в кодовой базе.

2. **Адаптация для дискретной оптимизации**:
   - Вместо непрерывных позиций волки перемещаются по узлам графа
   - Движение осуществляется к соседним узлам лидеров (альфа, бета, дельта)
   - Используется функция `get_all_neighbor_nodes` для определения соседей

3. **Основные параметры алгоритма**:
   - `max_iter`: максимальное количество итераций (по умолчанию 50)
   - `pack_size`: размер популяции волков (по умолчанию 20)
   - `appr_val`: доля узлов графа для приемлемого покрытия (по умолчанию 0.95)

4. **Механизм оптимизации**:
   - Инициализация популяции случайными допустимыми узлами графа
   - На каждой итерации определение трех лучших решений (альфа, бета, дельта)
   - Обновление позиций остальных волков через движение к соседям лидеров
   - Использование классических формул GWO для коэффициентов A и C
   - Баланс между исследованием (exploration) и эксплуатацией (exploitation)

5. **Интеграция с существующей архитектурой**:
   - Использование `NodeMetric` для оценки качества решений
   - Совместимость с тем же интерфейсом, что и другие алгоритмы
   - Поддержка параметров `area`, `points_list` и т.д.

#### Как работает алгоритм:

1. Инициализирует популяцию волков случайными узлами графа
2. На каждой итерации:
   - Определяет трех лучших волков (альфа, бета, дельта)
   - Для каждого остального волка:
     - Рассчитывает коэффициенты по формулам GWO
     - Формирует список соседей лидеров как кандидатов для перемещения
     - Перемещает волка к лучшему соседу, если это улучшает решение
3. Возвращает лучшее найденное решение

Эта реализация строго следует логике, установленной в решении, и интегрируется в существующую экосистему алгоритмов оптимизации размещения.

### Код

In [ ]:
import random

from collections import Counter

from genesis.core import BestPointsBase, StateBase, MetricBase
from genesis.states import FirstArrivalUnitState
from genesis.metrics import ArrivalTime, CoverIndex
from genesis.best_points import NodeMetric
from genesis.tools import get_all_neighbor_nodes

class BestNodeGWO(BestPointsBase):
    """
    Поиск лучших узлов графа с использованием алгоритма серых волков (Grey Wolf Optimizer, GWO).

    Алгоритм имитирует поведение стаи серых волков при охоте, где три лучших волка (альфа, бета, дельта)
    ведут остальных волков к оптимальному решению.

    Args:
        state_function (StateBase): Функция расчета состояния окружения.
        metric_function (MetricBase): Функция расчета метрики.
        appr_val (float, optional): Доля узлов графа для приемлемого покрытия. По умолчанию 0.95.
        max_iter (int, optional): Максимальное количество итераций. По умолчанию 50.
        pack_size (int, optional): Размер стаи волков. По умолчанию 20.
        **kwargs: Дополнительные параметры.
    """
    def __init__(self,
                 state_function:         StateBase,
                 metric_function:        MetricBase,
                 appr_val:               float    = 0.95,
                 max_iter:               int      = 20,
                 pack_size:              int      = 10,
                 hunt_bound:             float    = 1,
                 weight:                 str      = 'travel_time',
                 consensus_value:        float    = 0.75,
                 iteration_end_function: callable = None,
                 **kwargs) -> None:
        '''
        ## Аргументы
        `state_function`: StateBase
            функция расчета состояния окружения
        `metric_function`: MetricBase
            Функция расчета метрики
        `appr_val`: = 0.95
            Доля узлов графа, покрытие которой считается приемлемой для принятия расчетной метрики.
            Если при расчете метрик, из стартового узла (узлов) достижимо меньшее количество узлов,
            то такой узел не рассматривается.
        `max_iter`: int = 50
            Максимальное количество итераций алгоритма.
        `pack_size`: int = 20
            Размер стаи волков.
        `hunt_bound`: float = 1
            Порог начала фазы охоты.
        `weight`: str = 'travel_time'
            Имя поля содержащего вес ребер, или функция позволяющая вычислять 
            вес динамически.
        `consensus_value`: float = 0.75
            Количество волков, находящихся в одном месте (жертве),
            при котором считается, что алгоритм успешно завершен.
        `iteration_end_function`: callable = None
            Функция, вызываемая в конце каждой итерации.
        '''
        if hunt_bound < 0 or hunt_bound > 2:
            raise ValueError(f"Порог охоты не может выходить за пределы диапазона [0, 2]. Сейчас {hunt_bound}")
        if consensus_value < 0 or consensus_value > 1:
            raise ValueError(f"Порог охоты не может выходить за пределы диапазона [0, 1]. Сейчас {consensus_value}")

        self.appr_val = appr_val
        self.max_iter = max_iter
        self.pack_size = pack_size
        self.hunt_bound = hunt_bound
        self.node_metric_func = NodeMetric(state_function, metric_function, appr_val, err_val=None, **kwargs)
        self.weight = weight
        self.consensus_value = consensus_value
        self.iteration_end_function = iteration_end_function
        super().__init__(state_function, metric_function, **kwargs)


    def _find_closest_node_to_three_weighted(self, env, 
                                            nodes,
                                            weight='travel_time'):
        """
        Находит узел графа env, находящийся ближе всего к трем переданным узлам (для взвешенного графа).
        
        Параметры:
        env : networkx.Graph
            Взвешенный граф, в котором происходит поиск
        nodes : list
            Список из трех узлов графа
        weight : str, optional
            Название атрибута ребра, представляющего вес (по умолчанию 'travel_time')
            
        Возвращает:
        int или str
            Идентификатор узла, сумма расстояний от которого до трех заданных узлов минимальна
        """
        if len(nodes) != 3:
            raise ValueError("Должно быть передано ровно три узла")
        
        # Проверяем, что все узлы существуют в графе
        for node in nodes:
            if node not in env.nodes():
                raise ValueError(f"Узел {node} не существует в графе")
        
        # Вычисляем кратчайшие пути от каждого из трех узлов до всех остальных
        distances = {}
        for node in nodes:
            distances[node] = nx.single_source_dijkstra_path_length(env, node, weight=weight)
        
        # Находим узел с минимальной суммой расстояний до трех заданных узлов
        min_sum = float('inf')
        closest_node = None
        
        for node in env.nodes():
            # Сумма расстояний от текущего узла до трех заданных
            distance_sum = max(distances[source_node].get(node, float('inf')) for source_node in nodes)

            # Если сумма конечная (все пути существуют) и меньше текущего минимума
            if distance_sum < min_sum:
                min_sum = distance_sum
                closest_node = node
        
        return closest_node


    def _shortest_path_within_distance(self, env, source, target, max_distance, weight='travel_time'):
        """
        Находит кратчайший путь между исходной и целевой вершинами и возвращает
        только ту часть пути, которая находится в пределах максимального расстояния от источника.
        
        Параметры:
        env : networkx.Graph
            Граф для поиска
        source : node
            Начальная вершина
        target : node
            Целевая вершина
        max_distance : float
            Максимально допустимое расстояние от начальной вершины
        weight : str, optional
            Название атрибута ребра для весов (по умолчанию 'travel_time')
            
        Возвращает:
        list или None
            Частичный путь как список вершин (от источника до самой дальней вершины в пределах расстояния),
            или None, если путь не существует
        """
        try:
            # Найти кратчайший путь между источником и целью
            full_path = nx.shortest_path(env, source=source, target=target, weight=weight)
            
            # Вычислить совокупные расстояния от источника вдоль пути
            cumulative_distances = [0]  # Distance to source is 0
            for i in range(1, len(full_path)):
                prev_node = full_path[i-1]
                curr_node = full_path[i]
                # Получить данные ребра и извлечь вес
                edge_weight = env.get_edge_data(prev_node, curr_node, 0)[weight]
                cumulative_distances.append(cumulative_distances[-1] + edge_weight)
            
            # Найти часть пути в пределах max_distance
            truncated_path = []
            for i, distance in enumerate(cumulative_distances):
                if distance <= max_distance:
                    truncated_path.append(full_path[i])
                else:
                    break
            
            # Вернуть None, если нет вершин в пределах расстояния
            return truncated_path if truncated_path else None
            
        except nx.NetworkXNoPath:
            # Путь между источником и целью не существует
            return None


    def _search(self, env, cur_node, dist_to_prey, **kwargs):
        """
            Поиск узлов в заданном радиусе от текущего узла.
        """
        g_nodes = ox.graph_to_gdfs(env, edges=False)
        if not ox.projection.is_projected(g_nodes.crs):
            g_nodes = ox.projection.project_gdf(g_nodes)
        search_area = g_nodes.loc[[cur_node]].geometry.buffer(dist_to_prey)
        local_nodes = g_nodes[g_nodes.within(search_area.iloc[0])]
        return list(local_nodes.index)

    def __call__(self, 
                env: nx.Graph,
                area=None,
                start_point: int = None,
                points_list: set = None,
                **kwargs) -> tuple[int | None, float | None]:
        """
        Основной цикл алгоритма GWO для поиска оптимального размещения пожарного подразделения.

        Args:
            env (nx.Graph): Граф улично-дорожной сети
            area (pd.Series, optional): Маска узлов графа. Значениями True отмечены узлы графа - цели расчета леса Вороного.
            Если не указана, расчет производится для всех узлов графа.
            start_point (int, optional): Не используется.
            points_list (set, optional): Множество узлов-кандидатов. Если не указано, используются все узлы графа.

        Returns:
            tuple: (best_node, best_metric) — лучший узел и значение метрики.
        """
        if not isinstance(env, nx.Graph):
            raise TypeError("Тип переменной `env` должен быть Graph!")

        # Если списка узлов изначально не передано, рассматриваются все узлы графа
        nodes_list = list(points_list) if points_list is not None else list(env.nodes())

        if len(nodes_list) < 3:
            raise ValueError("Для работы алгоритма GWO необходимо минимум 3 узла-кандидата")

        # Инициализация стаи волков (случайные узлы графа)
        pack = []
        pack_metrics = []
        alpha_metric = None
        alpha_node = None

        # Заполняем популяцию случайными узлами
        for _ in range(self.pack_size):
            # Выбираем случайный узел, для которого можно рассчитать метрику
            node_metric = None
            attempts = 0
            max_attempts = min(100, len(nodes_list))  # Ограничение на количество попыток

            while node_metric is None and attempts < max_attempts:
                node = random.choice(nodes_list)
                node_metric = self.node_metric_func(env=env, node=node, area=area, **kwargs)
                attempts += 1

            if node_metric is not None:
                pack.append(node)
                pack_metrics.append(node_metric)

        # Если не удалось инициализировать достаточное количество особей
        if len(pack) < 3:
            raise ValueError("Не удалось инициализировать популяцию с достаточным количеством допустимых узлов")

        # Основной цикл оптимизации
        for iter_num in range(self.max_iter):
            # print(pack)
            # print(sorted(pack_metrics)[:3])
            print(f'{iter_num:<2}', end='')
            # Сортировка волков по значению метрики (лучшие первые)
            sorted_indices = sorted(range(len(pack_metrics)), 
                                  key=lambda i: pack_metrics[i])

            # Предполагаем, что лучшая метрика - минимальная (для времени прибытия)
            # Если лучшая метрика - максимальная (для индекса прикрытия), то разворачиваем список
            if self.metric_function.compare(1, 2) == 2:
                sorted_indices = sorted_indices[::-1]
            
            # Определение альфа, бета и дельта волков (три лучших решения)
            alpha_idx = sorted_indices[0]
            alpha_pos = pack[alpha_idx]
            
            beta_idx = sorted_indices[1]
            beta_pos = pack[beta_idx]
            
            delta_idx = sorted_indices[2]
            delta_pos = pack[delta_idx]

            # Определяем предполагаемую позицию жертвы
            prey_node = self._find_closest_node_to_three_weighted(
                    env = env,
                    nodes = [alpha_pos, beta_pos, delta_pos],
                    weight      = self.weight
            )

            # g_nodes = ox.graph_to_gdfs(env, edges=False)
            # print_temp_results(g_nodes, pack, [alpha_pos, beta_pos, delta_pos])
            # print_temp_results(iter_num, g_nodes, pack, [alpha_pos, beta_pos, delta_pos], prey_node)
            
            # Обновление позиций всех волков (кроме трех лидеров)
            a = 2 - iter_num * (2 / self.max_iter)  # Коэффициент уменьшающийся от 2 до 0
            # a = iter_num / self.max_iter
            
            for i, cur_node in enumerate(pack):
                # print('.', end='')
                

                # Пропускаем трех лидеров
                if cur_node in [alpha_pos, beta_pos, delta_pos]:
                    continue
                    
                # Расчет коэффициентов A и C для каждого волка
                r1 = random.random()
                r2 = random.random()
                
                A = 2 * a * r1 - a  # Коэффициент A
                # C = r2              # Коэффициент C
                dist_to_prey = nx.shortest_path_length(env, cur_node, prey_node, weight='length') * r2
                # print(a, A, C, dist_to_prey)

                if abs(A) > self.hunt_bound:
                    print('G', end='')
                    # глобальный поиск
                    candidates = self._search(
                        env = env, 
                        cur_node = cur_node, 
                        dist_to_prey = dist_to_prey,
                    )

                else:
                    # print(f'P,{cur_node},{prey_node}', end='')
                    print('P', end='')
                    # движение в сторону жертвы
                    candidates = self._shortest_path_within_distance(
                        env = env,
                        source = cur_node, 
                        target = prey_node, 
                        max_distance = dist_to_prey, 
                        weight='travel_time'
                    )

                    if candidates:
                        # разворачиваем маршрут, так, чтобы последний узел был первым
                        candidates = candidates[::-1]
                    else:
                        print('.', end='')
                        # Если путь найти не удалось, то выполняем глобальный поиск
                        candidates = self._search(
                            env = env, 
                            cur_node = cur_node, 
                            dist_to_prey = dist_to_prey,
                        )


                # print(candidates)
                if candidates:
                    new_pos = random.choice(candidates)
                    
                    # Проверяем, что новый узел допустим
                    new_metric = self.node_metric_func(env=env, node=new_pos, area=area, **kwargs)
                    
                    # Если метрика рассчитана успешно и она лучше текущей, обновляем позицию
                    if new_metric is not None:
                        if self.metric_function.compare(pack_metrics[i], new_metric) == new_metric:
                            pack[i] = new_pos
                            pack_metrics[i] = new_metric
                            # print(new_pos, new_metric)
                            if alpha_metric is None or self.metric_function.compare(alpha_metric, new_metric) == new_metric:
                                alpha_metric = new_metric
                                alpha_node = new_pos

            print(' ')
            if self.iteration_end_function:
                self.iteration_end_function(
                    value      = alpha_metric,                          # Лучшая метрика
                    abw_nodes = [alpha_pos, beta_pos, delta_pos],       # Позиция Альфы, Беты и Дельты
                    iteration  = iter_num,                              # Номер итерации
                    pack       = pack,                                  # Текущее положение волков
                    prey_node  = prey_node,                             # Текущая предположительная позиция жертвы
                    )
            
            # Если заданное количество волков нашло одну и ту же жертву, заканчиваем алгоритм
            if int(len(pack) * self.consensus_value) <= max(Counter(pack).values()):
                break
            
        # После завершения всех итераций находим лучшее решение
        best_idx = 0
        for i in range(1, len(pack_metrics)):
            if self.metric_function.compare(pack_metrics[best_idx], pack_metrics[i]) == pack_metrics[i]:
                best_idx = i
        
        return pack[best_idx], pack_metrics[best_idx]



state = FirstArrivalUnitState()
metric = ArrivalTime()

def iteration_end_func(value, abw_nodes, iteration, pack, prey_node):
    print(f'  Итерация:                         {iteration}')
    print(f'  Лучшая метрика:                   {value}')
    print(f'  Положение Лидеров:                {abw_nodes}')
    print(f'  Возможное положение лучшего узла: {prey_node}')
    print(f'  Стая:                             {pack}')
    consensus = max(Counter(pack).values())
    print(f'  Консенсус:                        {consensus}')



gwo = BestNodeGWO(
    state_function = state,
    metric_function = metric,
    appr_val = 0.95,
    max_iter = 10,
    pack_size = 10,
    consensus_value = 0.75,
    iteration_end_function = iteration_end_func
)

# Запуск GWO для одной точки размещения.
best_nodes_gwo, best_metric_gwo = gwo(
    env = G,
    area = None,
    start_point = None,
    points_list = None,
)



print('GWO Лучшее решение:', best_nodes_gwo)
print('GWO Метрика:', best_metric_gwo)

aco_times, _ = state(G, points=[best_nodes_gwo])
print('Среднее время прибытия GWO:', ArrivalTime()(aco_times))

In [ ]:
def print_temp_results(iter, g_nodes, nodes, best_nodes, prey_node, figsize=(8,8)):
    fig, ax = plt.subplots(1,1, figsize=figsize)
    g_nodes.plot(ax=ax, color='lightblue')
    g_nodes.loc[nodes].plot(ax=ax, color='red')
    g_nodes.loc[best_nodes].plot(ax=ax, color='green')
    # print(prey_node)
    if prey_node:
        g_nodes.loc[[prey_node]].plot(ax=ax, color='black')
    ax.axis('off')
    plt.savefig(f'_imgs/iter_{iter}.png')
    plt.close()

In [ ]:
def print_results(g_nodes, units, figsize=(10,10)):
    # Печать карты
    fig, ax = plt.subplots(1,1, figsize=figsize)
    g_nodes.query('arrival_time<=10').plot(ax=ax, color='g', label='Тпр <= 10', marker='s', legend=True)
    g_nodes.query('arrival_time<=20 and arrival_time>10').plot(ax=ax, color='y', label='Тпр > 10', marker='s', legend=True)
    g_nodes.query('arrival_time>20').plot(ax=ax, color='r', label='Тпр > 20', marker='s', legend=True)
    g_nodes.loc[units.keys()].plot(ax=ax, color='b', label='ПСЧ', marker='*', legend=True)
    ax.axis('off')
    ax.legend()
    plt.show()

    # Печать метрик
    print('     Среднее время прибытия: ', round(ArrivalTime()(times), 1), 'мин.')
    print('Максимальное время прибытия: ', round(ArrivalTime(np.max)(times), 1), 'мин.')
    print('                      ИП-10: ', CoverIndex(10)(g_nodes['arrival_time']), '%')
    print('                      ИП-20: ', CoverIndex(20)(g_nodes['arrival_time']), '%')

# Оценка результата
times, _ = FirstArrivalUnitState()(G, {best_nodes_gwo: 'full'})

g_nodes = ox.graph_to_gdfs(G, nodes=True, edges=False)
g_nodes['arrival_time'] = times

print_results(g_nodes, {best_nodes_gwo: 'full'}, figsize=(10,10))

# Тесты на реальных графах

In [ ]:
# 1. Загрузка исходных данных
## Загрузка ГДС
G = ox.load_graphml(r'tests\data\test_rng.ml')

## Упрощение графа (уменьшение количества узлов)
G = ox.simplification.simplify_graph(G)

## Загрузка зданий
buildings = gpd.read_file(r'tests\data\buildings.gpkg', layer='buildings')
### Сопоставление зданий с ближайшими узлами ГДС
centroids = ox.projection.project_gdf(buildings).geometry.centroid
buildings['node'] = ox.distance.nearest_nodes(ox.project_graph(G), 
                                              centroids.x,
                                              centroids.y
                                              )
# Дополняем набор весами
weights = [random.randint(1, 10) for _ in range(len(buildings))]
buildings['weight'] = weights

In [ ]:
def print_temp_results_2(g_nodes, pack, abw_nodes, prey_node, iter, figsize=(8,8)):
    fig, ax = plt.subplots(1,1, figsize=figsize)
    g_edges.plot(ax=ax, linewidth=0.2, color='grey', zorder=1)
    g_nodes.loc[pack].plot(ax=ax, marker='.', color='red', zorder=3)
    g_nodes.loc[[prey_node]].plot(ax=ax,
        marker='o',                  # Форма маркера (круг)
        facecolor='white',     # Цвет заливки (красный)
        edgecolor='black',     # Цвет границы (чёрный)
        # edgewidth=1,           # Толщина границы (тонкая)
        # markersize=20,               # Размер маркера
        zorder=2)
    g_nodes.loc[abw_nodes].plot(ax=ax, marker='*', color='green', zorder=4)
    ax.axis('off')
    plt.savefig(f'_imgs/iter_{iter}.png')
    plt.close()

In [ ]:
from fire_units.metrics import CoverIndexBuilding

state = FirstArrivalUnitState()
metric = CoverIndexBuilding(buildings, ip_val=10)
# metric = ArrivalTime()

g_nodes, g_edges = ox.graph_to_gdfs(G, nodes=True, edges=True)

def iteration_end_func(value, abw_nodes, iteration, pack, prey_node):
    print(f'  Итерация:                         {iteration}')
    print(f'  Лучшая метрика:                   {value}')
    print(f'  Положение лидеров:                {abw_nodes}')
    print(f'  Возможное положение лучшего узла: {prey_node}')
    print(f'  Стая:                             {pack}')
    consensus = max(Counter(pack).values())
    print(f'  Консенсус:                        {consensus}')
    print_temp_results_2(g_nodes, pack, abw_nodes, prey_node, iteration)



gwo = BestNodeGWO(
    state_function = state,
    metric_function = metric,
    appr_val = 0.95,
    max_iter = 30,
    pack_size = 30,
    hunt_bound = 0.5,
    consensus_value = 0.75,
    iteration_end_function = iteration_end_func
)

# Запуск GWO для одной точки размещения.
best_nodes_gwo, best_metric_gwo = gwo(
    env = G,
    area = None,
    start_point = None,
    points_list = None,
)



print('GWO Лучшее решение:', best_nodes_gwo)
print('GWO Метрика:', best_metric_gwo)

aco_times, _ = state(G, points=[best_nodes_gwo])
print('Среднее время прибытия GWO:', ArrivalTime()(aco_times))

In [ ]:
# Оценка результата
times, _ = FirstArrivalUnitState()(G, {best_nodes_gwo: 'gwo'})

g_nodes = ox.graph_to_gdfs(G, nodes=True, edges=False)
g_nodes['arrival_time'] = times

print_results(g_nodes, {best_nodes_gwo: 'gwo'}, figsize=(10,10))

### Большой граф

In [ ]:
import osmnx as ox
import matplotlib.pyplot as plt

from graphs.speeds import kmh_to_mm, set_graph_travel_times
from fire_units.settings import DEFAULT_SPEEDS

ox.settings.log_console         = True
ox.settings.use_cache           = True
ox.settings.overpass_url        = 'https://app-sibpsa.ru/overpass/api'
# ox.settings.overpass_url        = 'https://overpass.sibpsa.ru/api'
# ox.settings.overpass_url        = 'https://overpass.private.coffee/api'
# ox.settings.overpass_url        = 'https://overpass-api.de/api'
ox.settings.overpass_rate_limit = False         # Снимаем ограничение на количество запросов
ox.settings.default_access      = ''            # Разрешаем выгружать данные частных дорог

BUILDING_FIELDS   = {
                'name', 'amenity', 'building', 'addr:street', 'addr:housenumber',
                'building:levels', 'building:flats', 'geometry'
            }

In [ ]:
place        = 'Томск, Томская область, Российская Федерация'
network_type = 'drive_service'
buffer_size  = 500
simplify     = True
osmid        = None
    

if osmid:
    area = ox.geocode_to_gdf(osmid, by_osmid=True)
else:
    area = ox.geocode_to_gdf(place)
area_buffer = area.copy()
area_buffer = ox.projection.project_gdf(area_buffer)
area_buffer.geometry = area_buffer.geometry.buffer(buffer_size)
area_buffer = ox.projection.project_gdf(area_buffer, to_latlong=True)


# Выгрузка графа
G = ox.graph_from_polygon(area_buffer.union_all(),
                        network_type = network_type,
                        simplify     = False,
                        retain_all   = False,
)
G.name = place
print(G.number_of_nodes(), G.number_of_edges())

# Устанавливаем скорости DEFAULT_SPEEDS
set_graph_travel_times(G, speeds=DEFAULT_SPEEDS, morph_function=kmh_to_mm)
if simplify:
    G = ox.simplify_graph(G)
    print(G.number_of_nodes(), G.number_of_edges())


# Выгрузка зданий
tags = {'building': True}
buildings = ox.features_from_polygon(area.union_all(), tags)
print('Зданий:', len(buildings))

# Оставляем только полигоны и мультиполигоны
buildings = buildings[buildings.geometry.apply(lambda x: x.geom_type).isin(['Polygon', 'MultiPolygon'])]

# Оставляем только основные поля, если они есть
fields = list(set(buildings.columns) & set(BUILDING_FIELDS))
buildings = buildings[fields]

# Сброс индекса и удаление лишних столбцов, если есть
buildings.reset_index(inplace=True)
if 'element' in buildings.columns:
    buildings = buildings.drop('element', axis=1)
else:
    print('Поле "element" отсутствует в наборе полученных данных.')
if 'id' in buildings.columns:
    buildings = buildings.drop_duplicates('id')
    buildings = buildings.set_index('id')
else:
    print('Поле "id" отсутствует в наборе полученных данных.')


# Установка ближайших узлов зданий
g_nodes = ox.graph_to_gdfs(G, edges=False)
g_nodes = ox.projection.project_gdf(g_nodes)
buildings = ox.projection.project_gdf(buildings)
buildings = buildings.sjoin_nearest(g_nodes[['geometry']], max_distance=100)
# buildings.rename(columns={'osmid_right': 'node'}, inplace=True)
buildings.rename(columns={'osmid': 'node', 'osmid_right': 'node'}, inplace=True)
g_nodes = ox.projection.project_gdf(g_nodes, to_latlong=True)
buildings = ox.projection.project_gdf(buildings, to_latlong=True)

# Возвращаем результат
buildings.head(2)

In [ ]:
fig, ax = ox.plot_graph(G, edge_linewidth=0.2, node_size=0, show=False)
area.boundary.plot(ax=ax, color='r', lw=0.5)
buildings.plot(ax=ax)

plt.show()

In [ ]:
from fire_units.metrics import CoverIndexBuilding
from genesis.utils import timing

state = FirstArrivalUnitState()
metric = CoverIndexBuilding(buildings, ip_val=10)
# metric = ArrivalTime()

g_nodes, g_edges = ox.graph_to_gdfs(G, nodes=True, edges=True)

def iteration_end_func(value, abw_nodes, iteration, pack, prey_node):
    print(f'  Итерация:                         {iteration}')
    print(f'  Лучшая метрика:                   {value}')
    print(f'  Положение лидеров:                {abw_nodes}')
    print(f'  Возможное положение лучшего узла: {prey_node}')
    # print(f'  Стая:                             {pack}')
    consensus = max(Counter(pack).values())
    print(f'  Консенсус:                        {consensus}')
    # print_temp_results_2(g_nodes, pack, abw_nodes, prey_node, iteration)



gwo = timing(BestNodeGWO(
    state_function = state,
    metric_function = metric,
    appr_val = 0.95,
    max_iter = 30,
    pack_size = 30,
    hunt_bound = 0.5,
    consensus_value = 0.75,
    iteration_end_function = iteration_end_func
))

# Запуск GWO для одной точки размещения.
best_nodes_gwo, best_metric_gwo = gwo(
    env = G,
    area = None,
    start_point = None,
    points_list = None,
)



print('GWO Лучшее решение:', best_nodes_gwo)
print('GWO Метрика:', best_metric_gwo)

aco_times, _ = state(G, points=[best_nodes_gwo])
print('Среднее время прибытия GWO:', ArrivalTime()(aco_times))

In [ ]:
# 2. Сборка решения
## Выбор алгоритма моделирования 
## (алгоритмы расчета расчета кратчайшего пути
## и времени следования установлены по умолчанию)
state = FirstArrivalUnitState()

# 3. Моделирование параметров прибытия
times, nearest = state(env=G, points=[best_nodes_gwo])

# 4. Анализ результатов
# Дополнение данных о зданиях данными результатами моделирования (по полю `node`)
buildings_s = buildings.copy()
buildings_s = buildings_s.merge(times, left_on='node', right_index=True)
buildings_s = buildings_s.merge(nearest, left_on='node', right_index=True)

print('Результатов моделирования:', len(times))
print('Количество зданий:', len(buildings_s))

In [ ]:
# Подготовка холста
fig, ax = plt.subplots(1, 1, figsize=(16,8), tight_layout=True)

g_nodes = ox.graph_to_gdfs(G, nodes=True, edges=False)

# Карта 10 минутной зоны
area.boundary.plot(ax=ax, color='r', lw=0.5)
ox.plot_graph(G, node_size=0, edge_linewidth=0.2, edge_color='gray', ax=ax, close=False, show=False)
buildings_s[buildings_s['times']<=10].plot(ax=ax, color='g', edgecolor='g', alpha=0.5) 
buildings_s[buildings_s['times']>10].plot(ax=ax, color='r', edgecolor='r', alpha=0.5)
g_nodes.loc[[best_nodes_gwo]].plot(ax=ax, 
    color='red',
    facecolor='red',     # Цвет заливки (красный)
    edgecolor='black',     # Цвет границы (чёрный)
    markersize=50, 
    marker='o', 
    label='Пожарные подразделения')
ax.legend()


plt.show()